# Code for Basmati and Non-basmati rice data analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC

# -----------------------------
# Set Times New Roman font
# -----------------------------
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("grain_features_Basmati_corrected.csv")

colors = {
    "PB 1121": "#4E79A7",
    "PB 1401": "#F28E2B",
    "PR 14": "#59A14F"
}

# -----------------------------
# Create subplot figure
# -----------------------------
fig, axes = plt.subplots(2,2, figsize=(14,10))

# =====================================================
# (a) Length distribution
# =====================================================
ax = axes[0,0]

for cls in df["class"].unique():
    
    subset = df[df["class"] == cls]
    
    ax.hist(
        subset["length_mm"],
        bins=40,
        alpha=0.6,
        label=cls,
        color=colors.get(cls)
    )

ax.set_xlabel("Length (mm)", fontsize=18, fontweight="bold")
ax.set_ylabel("Frequency", fontsize=18, fontweight="bold")

ax.tick_params(labelsize=14)
ax.set_title("(a)", fontsize=18, fontweight="bold")


# =====================================================
# (b) Aspect ratio distribution
# =====================================================
ax = axes[0,1]

for cls in df["class"].unique():
    
    subset = df[df["class"] == cls]
    
    ax.hist(
        subset["aspect_ratio"],
        bins=40,
        alpha=0.6,
        label=cls,
        color=colors.get(cls)
    )

ax.set_xlabel("Aspect Ratio (Length / Width)", fontsize=18, fontweight="bold")
ax.set_ylabel("Frequency", fontsize=18, fontweight="bold")

ax.tick_params(labelsize=14)
ax.set_title("(b)", fontsize=18, fontweight="bold")


# =====================================================
# (c) KDE Aspect Ratio
# =====================================================
ax = axes[1,0]

sns.kdeplot(
    data=df,
    x="aspect_ratio",
    hue="class",
    fill=True,
    common_norm=False,
    ax=ax
)

# Remove legend from KDE subplot
ax.legend_.remove()

ax.set_xlabel("Aspect Ratio (Length / Width)", fontsize=18, fontweight="bold")
ax.set_ylabel("Density", fontsize=18, fontweight="bold")

ax.tick_params(labelsize=14)
ax.set_title("(c)", fontsize=18, fontweight="bold")


# =====================================================
# (d) Morphological decision map
# =====================================================
ax = axes[1,1]

df["label"] = df["class"].apply(
    lambda x: "Basmati" if x in ["PB 1121","PB 1401"] else "Adulterant"
)

X = df[["length_mm","aspect_ratio"]]
y = (df["label"]=="Adulterant").astype(int)

model = SVC(kernel="linear")
model.fit(X,y)

x_min, x_max = X["length_mm"].min()-0.5, X["length_mm"].max()+0.5
y_min, y_max = X["aspect_ratio"].min()-0.5, X["aspect_ratio"].max()+0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.2, cmap="coolwarm")

for cls in df["class"].unique():
    
    subset = df[df["class"] == cls]
    
    ax.scatter(
        subset["length_mm"],
        subset["aspect_ratio"],
        color=colors.get(cls),
        label=cls,
        s=20,
        alpha=0.7
    )

ax.set_xlabel("Length (mm)", fontsize=18, fontweight="bold")
ax.set_ylabel("Aspect Ratio", fontsize=18, fontweight="bold")

ax.tick_params(labelsize=14)
ax.set_title("(d)", fontsize=18, fontweight="bold")


# =====================================================
# Make tick labels bold
# =====================================================
for ax in axes.flat:
    
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')
        
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')


# =====================================================
# Remove grid from all plots
# =====================================================
for ax in axes.flat:
    ax.grid(False)


# =====================================================
# Global legend
# =====================================================
handles, labels = axes[0,0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=3,
    fontsize=14,
    frameon=False
)

plt.tight_layout(rect=[0,0.05,1,1])


# -----------------------------
# Save figure
# -----------------------------
plt.savefig("morphological_feature_analysis.png", dpi=600)
plt.savefig("morphological_feature_analysis.pdf", dpi=600)

plt.show()